In [22]:
# Raw Data Loading
import pandas as pd

df_target = pd.read_csv('./report.txt', sep='\t')
df_target.head()

,기간,구분,구분.1,전체,남자,여자
0,2009,서울시,서울시,24.3,45.5,4
1,2009,생애주기별,19~29세,24.9,43.8,6
2,2009,생애주기별,30~44세,29.9,55.1,3.8
3,2009,생애주기별,45~64세,22.6,43.2,3.3
4,2009,생애주기별,65세 이상,11.5,22.8,2.9


In [23]:
df_target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   기간      429 non-null    int64  
 1   구분      429 non-null    object 
 2   구분.1    429 non-null    object 
 3   전체      429 non-null    float64
 4   남자      429 non-null    float64
 5   여자      429 non-null    object 
dtypes: float64(2), int64(1), object(3)
memory usage: 20.2+ KB


In [24]:
# 컬럼명 수정
# 구분1의 서울시를 시로 수정
# 구분1의 **구를 구로 수정 (ex. 강남구, 영등포구 -> 구)
tmp_df = df_target.copy()

tmp_df = tmp_df.rename(columns={'구분': '구분1', '구분.1': '구분2'})

tmp_df['구분1'] = tmp_df['구분1'].replace('서울시', '시')
tmp_df['구분1'] = tmp_df['구분1'].str.replace(r'.*구$', '구', regex=True)

df_target = tmp_df.copy()
df_target.head()

,기간,구분1,구분2,전체,남자,여자
0,2009,시,서울시,24.3,45.5,4
1,2009,생애주기별,19~29세,24.9,43.8,6
2,2009,생애주기별,30~44세,29.9,55.1,3.8
3,2009,생애주기별,45~64세,22.6,43.2,3.3
4,2009,생애주기별,65세 이상,11.5,22.8,2.9


In [25]:
# 여자 컬럼의 데이터중 float type으로 변경할 수 있는 경우 float로 변경
# 변경할 수 없는 경우 NaN으로 변경
def convert_str_to_float(value):
    try:
        return float(value)
    except Exception as e:
        return None

tmp_df = df_target.copy()

tmp_df['여자'] = tmp_df['여자'].apply(convert_str_to_float)

df_target = tmp_df.copy()
df_target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 429 entries, 0 to 428
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   기간      429 non-null    int64  
 1   구분1     429 non-null    object 
 2   구분2     429 non-null    object 
 3   전체      429 non-null    float64
 4   남자      429 non-null    float64
 5   여자      379 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 20.2+ KB


In [26]:
df_target['구분1'].unique()

array(['시', '생애주기별', '교육수준별Ⅰ(30~64세)', '교육수준별Ⅱ(65세 이상)', '직업별(30~64세)',
       '구'], dtype=object)

In [27]:
# 구분1 컬럼의 교육수준별Ⅱ(65세 이상) 데이터는 결측이 많아 삭제
tmp_df = df_target.copy()

tmp_df = tmp_df[~(tmp_df['구분1'] == '교육수준별Ⅱ(65세 이상)')]

df_target = tmp_df.copy()
df_target.info()

<class 'pandas.core.frame.DataFrame'>
Index: 396 entries, 0 to 428
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   기간      396 non-null    int64  
 1   구분1     396 non-null    object 
 2   구분2     396 non-null    object 
 3   전체      396 non-null    float64
 4   남자      396 non-null    float64
 5   여자      371 non-null    float64
dtypes: float64(3), int64(1), object(2)
memory usage: 21.7+ KB


$$
x = 100 \times \frac{전체 - 남자}{여자 - 남자}
$$

In [28]:
# 여자 컬럼의 결측을 채우고자 여자비율을 구해
# 여자비율의 변화를 확인하여 결측을 채움
# 이를 위해 여자비율 컬럼 생성
def get_female_ratio(df):
    if df['여자'] is None:
        return None
    else:
        x = (df['전체'] - df['남자']) / (df['여자'] - df['남자']) * 100
        return round(x, 6)

tmp_df = df_target.copy()


tmp_df['여자비율'] = tmp_df.apply(get_female_ratio, axis=1)

df_target = tmp_df.copy()
df_target.head()

,기간,구분1,구분2,전체,남자,여자,여자비율
0,2009,시,서울시,24.3,45.5,4.0,51.084337
1,2009,생애주기별,19~29세,24.9,43.8,6.0,50.000000
2,2009,생애주기별,30~44세,29.9,55.1,3.8,49.122807
3,2009,생애주기별,45~64세,22.6,43.2,3.3,51.629073
4,2009,생애주기별,65세 이상,11.5,22.8,2.9,56.783920


In [29]:
# 여자 비율에 결측이 있는 구 목록
tmp_df = df_target.copy()

df_gu = tmp_df[tmp_df['구분1'] == '구']
targets = sorted(df_gu.loc[df_gu['여자비율'].isna(), '구분2'].dropna().unique())
targets

['강동구',
 '구로구',
 '노원구',
 '도봉구',
 '동대문구',
 '동작구',
 '서대문구',
 '서초구',
 '성동구',
 '송파구',
 '양천구',
 '영등포구']

In [30]:
# 여자 컬럼의 결측을 채우기 위해서 여자비율의 회귀계수, MinMax, 평균, 표준편차 구하기
# 이를 위해 여자비율 컬럼에 결측이 있는 구만을 추출한 후,
# 이들 중 결측이 없는 데이터만 사용하여 여러 정보를 구함
import numpy as np

tmp_df = df_target.copy()

df_gu = tmp_df[tmp_df['구분1'] == '구']
targets = sorted(df_gu.loc[df_gu['여자비율'].isna(), '구분2'].dropna().unique())
tmp_df = tmp_df[tmp_df['구분2'].isin(targets)].dropna()

result = []
for gu, group in tmp_df.groupby('구분2'):
    x = group['기간'].values
    y = group['여자비율'].values

    w, b = np.polyfit(x, y, 1)

    minmax = np.abs(y.max() - y.min())
    mean = np.abs(y.mean())
    std = np.abs(y.std())

    result.append({
        '구': gu,
        '회귀계수': round(w, 6),
        'MinMax': round(minmax, 6),
        '평균': round(mean, 6),
        '표준편차': round(std, 6)
    })

df_result = pd.DataFrame(result)
df_result = df_result.sort_values('구').reset_index(drop=True)

df_result.head()

,구,회귀계수,MinMax,평균,표준편차
0,강동구,0.052052,0.701401,50.631950,0.222337
1,구로구,0.085093,1.011929,50.267907,0.293120
2,노원구,0.016932,0.400031,52.033941,0.153785
3,도봉구,0.024619,0.580787,51.265114,0.148100
4,동대문구,0.042682,0.488017,50.060631,0.141627


In [33]:
# 위에서 구한 정보를 이용하여 여자비율의 결측 채우기
# 우선 회귀계수의 경우 대부분 양수지만 매우 작은 값을 가지고 있음
# MinMax의 경우 최대 여자비율과 최소 여자비율의 차가 가장 큰 부분도 1.2%p 미만 수준임
# 표준편차의 경우 대부분 평균의 0.5%p 미만 수준으로 
# 평균에 대부분의 여자비율 Data가 몰려있다고 볼 수 있음
# 따라서 여자비율은 평균에서 큰 차이가 없다고 보여져 결측을 평균으로 채움
mean_map = df_result.set_index('구')['평균']

df_target['여자비율'] = df_target['여자비율'].fillna(
    df_target['구분2'].map(mean_map).round(6)
)
df_target.head()

,기간,구분1,구분2,전체,남자,여자,여자비율
0,2009,시,서울시,24.3,45.5,4.0,51.084337
1,2009,생애주기별,19~29세,24.9,43.8,6.0,50.000000
2,2009,생애주기별,30~44세,29.9,55.1,3.8,49.122807
3,2009,생애주기별,45~64세,22.6,43.2,3.3,51.629073
4,2009,생애주기별,65세 이상,11.5,22.8,2.9,56.783920


$$
여자 = \frac{전체 - (1 - \frac{여자비율}{100} \times 남자)}{여자비율 / 100}
$$

In [34]:
# 위 단계에서 처리한 여자비율 컬럼을 사용하여 여자 컬럼 결측 채우기
tmp_df = df_target.copy()

mask = df_target['여자'].isna()
tmp_df.loc[mask, '여자'] = (
    (tmp_df.loc[mask, '전체'] - (1 - tmp_df.loc[mask, '여자비율']/100) * tmp_df.loc[mask, '남자'])
    / (tmp_df.loc[mask, '여자비율']/100)
).round(1)

tmp_df = tmp_df.drop(columns=['여자비율'])

df_target = tmp_df.copy()
df_target.head()

,기간,구분1,구분2,전체,남자,여자
0,2009,시,서울시,24.3,45.5,4.0
1,2009,생애주기별,19~29세,24.9,43.8,6.0
2,2009,생애주기별,30~44세,29.9,55.1,3.8
3,2009,생애주기별,45~64세,22.6,43.2,3.3
4,2009,생애주기별,65세 이상,11.5,22.8,2.9


In [35]:
# 구분2별로 전체, 남자, 여자 컬럼의 데이터 단조 증가, 감소 파악
tmp_df = df_target.copy()

result = []
for (part1, part2), group in tmp_df.groupby(['구분1', '구분2']):
    row = {
        '구분1': part1,
        '구분2': part2
    }
    for col in ['전체', '남자', '여자']:
        s = group[col]

        is_monotonic = s.is_monotonic_increasing or s.is_monotonic_decreasing
        row[col] = is_monotonic
    result.append(row)

df_result = pd.DataFrame(result)
df_result = df_result.sort_values(['구분1', '구분2']).reset_index(drop=True)
df_result.head()

,구분1,구분2,전체,남자,여자
0,교육수준별Ⅰ(30~64세),고졸,False,False,False
1,교육수준별Ⅰ(30~64세),대졸이상,False,False,False
2,교육수준별Ⅰ(30~64세),중졸이하,False,False,False
3,구,강남구,False,False,False
4,구,강동구,False,False,False


In [36]:
# 구분2별로 전체, 남자, 여자 흡연률이 해가 지남에 따라
# 증가 또는 감소하는지 회귀계수를 통해 파악
def slopes_by_group(group: pd.DataFrame) -> pd.Series:
    x = group['기간'].to_numpy()
    result = {}
    for col in ['전체', '남자', '여자']:
        y = group[col].to_numpy()
        if x.size >= 2:
            result[col] = np.round(np.polyfit(x, y, 1)[0], 6)
        else:
            result[col] = np.nan
    return pd.Series(result, index=['전체', '남자', '여자'])

df_result = (
df_target
.groupby(['구분1', '구분2'], dropna=False)
.apply(slopes_by_group, include_groups=False)
.reset_index()
.sort_values(['구분1', '구분2'])
.reset_index(drop=True)
)
df_result.head()

,구분1,구분2,전체,남자,여자
0,교육수준별Ⅰ(30~64세),고졸,-0.367273,-0.797273,0.074545
1,교육수준별Ⅰ(30~64세),대졸이상,-0.872727,-1.285455,-0.013636
2,교육수준별Ⅰ(30~64세),중졸이하,0.006364,-0.205455,0.075455
3,구,강남구,-0.488182,-0.910909,-0.124545
4,구,강동구,-0.909091,-1.543636,-0.260000


In [37]:
# 구분1별 전체 흡연률 감소세가 최대/최소인 구분2 데이터
tmp_df = df_result.copy()

result = []
for part1, group in tmp_df.groupby('구분1'):
    if part1 == '시':
        continue
    
    neg_group = group[group['전체'] < 0]
    
    if len(neg_group) == 0:
        continue
    
    max_row = neg_group.loc[neg_group['전체'].idxmax(), '구분2']
    min_row = None
    
    if len(neg_group) > 1:
        min_row = neg_group.loc[neg_group['전체'].idxmin(), '구분2']  

    result.append({
        '구분1': part1,
        '최대': max_row,
        '최소': min_row
    })

df_maxmin = pd.DataFrame(result)
df_maxmin = df_maxmin.sort_values('구분1').reset_index(drop=True)
df_maxmin.head()

,구분1,최대,최소
0,교육수준별Ⅰ(30~64세),고졸,대졸이상
1,구,중랑구,강동구
2,생애주기별,45~64세,30~44세
3,직업별(30~64세),육체직,사무직


In [ ]:
# 구분1별 여자 흡연률 증가세가 최대/최고인 구분2 데이터
tmp_df = df_result.copy()

result = []
for part1, group in tmp_df.groupby('구분1'):
    if part1 == '시':
        continue
    
    pos_group = group[group['여자'] > 0]
    
    if len(pos_group) == 0:
        continue
    
    max_row = pos_group.loc[pos_group['여자'].idxmax(), '구분2']
    min_row = None
    
    if len(pos_group) > 1:
        min_row = pos_group.loc[pos_group['여자'].idxmin(), '구분2']

    result.append({
        '구분1': part1,
        '최대': max_row,
        '최소': min_row
    })

df_maxmin = pd.DataFrame(result)
df_maxmin = df_maxmin.sort_values('구분1').reset_index(drop=True)
df_maxmin

,구분1,최대,최소
0,교육수준별Ⅰ(30~64세),중졸이하,고졸
1,구,은평구,성동구
2,생애주기별,19~29세,None
3,직업별(30~64세),서비스ㆍ판매직,None
